In [ ]:
! pip install -q kaggle

In [ ]:
! mkdir ~/.kaggle

mkdir: cannot create directory ‘/root/.kaggle’: File exists


In [ ]:
! cp kaggle.json ~/.kaggle/

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
! kaggle datasets list

ref                                                                    title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
jayaantanaath/student-habits-vs-academic-performance                   Student Habits vs Academic Performance                   19512  2025-04-12 10:49:08.663000           5025         86  1.0              
adilshamim8/student-depression-dataset                                 Student Depression Dataset                              467020  2025-03-13 03:12:30.423000          21701        346  1.0              
zahidmughal2343/supplement-sales-data                                  Supplement Sales Data                                    66800  2025-04-11 15:01:00.227000           

In [ ]:
!kaggle datasets download -d 'nicopalv/dataset-klasifikasi-gambar-hewan'

Dataset URL: https://www.kaggle.com/datasets/nicopalv/dataset-klasifikasi-gambar-hewan
License(s): unknown
dataset-klasifikasi-gambar-hewan.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
import zipfile

In [ ]:
dataset_zip = zipfile.ZipFile('dataset-klasifikasi-gambar-hewan.zip', 'r')

dataset_zip.extractall()

dataset_zip.close()

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
# PATH
base_dir = 'dataset'
train_dir = os.path.join(base_dir, 'training_set')
test_dir = os.path.join(base_dir, 'test_set')

In [ ]:
# Image Preprocessing
img_height, img_width = 150, 150
batch_size = 32

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True
)

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

Found 9600 images belonging to 3 classes.


In [ ]:
validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)

Found 2400 images belonging to 3 classes.


In [ ]:
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Found 3000 images belonging to 3 classes.


In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(256, (3,3), activation='relu'),  # Tambahan
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 kelas
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

In [ ]:
model.compile(optimizer=optimizer,
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 15, 15, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     3,211,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,600,707 (13.74 MB)

 Trainable params: 3,600,707 (13.74 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, verbose=1)
]

In [ ]:
# Training
epochs = 50
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator,
    callbacks=callbacks
)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4450 - loss: 1.0179

300/300 ━━━━━━━━━━━━━━━━━━━━ 662s 2s/step - accuracy: 0.4453 - loss: 1.0176 - val_accuracy: 0.6092 - val_loss: 0.7610 - learning_rate: 1.0000e-04
Epoch 2/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6373 - loss: 0.7477

300/300 ━━━━━━━━━━━━━━━━━━━━ 690s 2s/step - accuracy: 0.6374 - loss: 0.7476 - val_accuracy: 0.6804 - val_loss: 0.6763 - learning_rate: 1.0000e-04
Epoch 3/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6556 - loss: 0.6822

300/300 ━━━━━━━━━━━━━━━━━━━━ 641s 2s/step - accuracy: 0.6557 - loss: 0.6822 - val_accuracy: 0.7154 - val_loss: 0.6100 - learning_rate: 1.0000e-04
Epoch 4/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6874 - loss: 0.6312

...

300/300 ━━━━━━━━━━━━━━━━━━━━ 665s 2s/step - accuracy: 0.7829 - loss: 0.4552 - val_accuracy: 0.7979 - val_loss: 0.4203 - learning_rate: 1.0000e-04
Epoch 13/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 631s 2s/step - accuracy: 0.8047 - loss: 0.4237 - val_accuracy: 0.7958 - val_loss: 0.4180 - learning_rate: 1.0000e-04
Epoch 14/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8078 - loss: 0.4220

300/300 ━━━━━━━━━━━━━━━━━━━━ 633s 2s/step - accuracy: 0.8078 - loss: 0.4220 - val_accuracy: 0.8092 - val_loss: 0.4018 - learning_rate: 1.0000e-04
Epoch 15/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 633s 2s/step - accuracy: 0.8100 - loss: 0.4250 - val_accuracy: 0.7900 - val_loss: 0.4448 - learning_rate: 1.0000e-04
Epoch 16/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8095 - loss: 0.4150

300/300 ━━━━━━━━━━━━━━━━━━━━ 639s 2s/step - accuracy: 0.8095 - loss: 0.4150 - val_accuracy: 0.8133 - val_loss: 0.4085 - learning_rate: 1.0000e-04
Epoch 17/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8194 - loss: 0.4021

300/300 ━━━━━━━━━━━━━━━━━━━━ 631s 2s/step - accuracy: 0.8194 - loss: 0.4021 - val_accuracy: 0.8167 - val_loss: 0.3988 - learning_rate: 1.0000e-04
Epoch 18/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 638s 2s/step - accuracy: 0.8242 - loss: 0.3959 - val_accuracy: 0.8075 - val_loss: 0.4104 - learning_rate: 1.0000e-04
Epoch 19/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8271 - loss: 0.3802

300/300 ━━━━━━━━━━━━━━━━━━━━ 637s 2s/step - accuracy: 0.8271 - loss: 0.3802 - val_accuracy: 0.8296 - val_loss: 0.3775 - learning_rate: 1.0000e-04
Epoch 20/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 639s 2s/step - accuracy: 0.8285 - loss: 0.3754 - val_accuracy: 0.8000 - val_loss: 0.4019 - learning_rate: 1.0000e-04
Epoch 21/50
300/300 ━━━━━━━━━━━━━━━━━━━━ 671s 2s/step - accuracy: 0.8268 - loss: 0.3990 - val_accuracy: 0.8046 - val_loss: 0.4045 - learning_rate: 1.0000e-04
Epoch 22/50
288/300 ━━━━━━━━━━━━━━━━━━━━ 23s 2s/step - accuracy: 0.8261 - loss: 0.3832

In [ ]:
# Evaluation
loss, accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {accuracy*100:.2f}%")

In [ ]:
# Plot Accuracy and Loss
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

In [ ]:
epochs_range = range(len(acc))

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Train Acc', marker='o')
plt.plot(epochs_range, val_acc, label='Val Acc', marker='o')
plt.legend()
plt.title('Accuracy')
plt.grid()

In [ ]:
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Train Loss', marker='o')
plt.plot(epochs_range, val_loss, label='Val Loss', marker='o')
plt.legend()
plt.title('Loss')
plt.grid()

In [ ]:
plt.tight_layout()
plt.show()

Save Model

In [ ]:
# SavedModel
model.save("saved_model/image_classifier")


In [ ]:
# TFLite
converter = tf.lite.TFLiteConverter.from_saved_model("saved_model/image_classifier")
tflite_model = converter.convert()
with open("image_classifier.tflite", "wb") as f:
    f.write(tflite_model)

In [ ]:
# TFJS
!pip install tensorflowjs
import tensorflowjs as tfjs
tfjs.converters.save_keras_model(model, "tfjs_model")